# Knee MRI: error analysis of the baseline OOF predictions
CPU only (Accelerator: None). Pandas + scikit-learn, takes a minute.
**Inputs to attach:** baseline training output (`knee-mri-diagnosis-training`, has `oof_all.csv`), caching output (`knee-mri-diagnosis-caching`, has `cache_index.csv`), the competition data (for `train.csv`, optional).
Run top to bottom. Cell 1 prints what it found and the column names; if it refuses to guess, set the `*_OVERRIDE` variables.
Cell 2 also derives field strength (from `site_proxy`) and report language (from the text; `pip install langdetect` with internet ON is better than the script-only fallback). Cells 4 and 4c then break the errors down by those. Tested on synthetic data only.

In [1]:
# 0. config
import os, glob, re
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 120)

IN = "/kaggle/input"
TRAIN_NB = f"{IN}/notebooks/theokorir/knee-mri-diagnosis-training"
CACHE_NB = f"{IN}/notebooks/theokorir/knee-mri-diagnosis-caching"
DATA     = f"{IN}/competitions/rsna-knee-abnormality-detection"
OUT = "/kaggle/working/error_analysis"; os.makedirs(OUT, exist_ok=True)

# overrides (leave None unless cell 1 asks)
OOF_OVERRIDE = None          # path to oof_all.csv
META_OVERRIDE = None         # path to cache_index.csv (site / vendor columns)
TRAIN_CSV_OVERRIDE = None    # path to competition train.csv (report text)
PRED_PREFIX_OVERRIDE = None  # e.g. "p_"
GROUP_COLS_OVERRIDE = None   # e.g. ["vendor", "site_proxy"]
LABEL = "ACL"                # label to inspect in detail (cells 5-6)

for d in (TRAIN_NB, CACHE_NB, DATA):
    print(d, "->", sorted(os.listdir(d))[:12] if os.path.isdir(d) else "MISSING")

/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training -> ['__huggingface_repos__.json', '__notebook__.ipynb', '__output__.json', '__results__.html', 'custom.css', 'models_g0', 'models_g0.log', 'models_g1', 'models_g1.log', 'oof_all.csv']
/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching -> ['__notebook__.ipynb', '__output__.json', '__results__.html', '__results___files', 'cache', 'custom.css', 'labels_soft.csv', 'reliability.csv']
/kaggle/input/competitions/rsna-knee-abnormality-detection -> ['sample_submission.csv', 'test.csv', 'test_series', 'test_series.csv', 'train.csv', 'train_series', 'train_series.csv']


In [2]:
# 1. load OOF predictions and find columns
def pick(cands, override=None, what=""):
    if override: return override
    found = [c for c in cands if os.path.exists(c)]
    print(what, "candidates found:", found)
    if len(found) != 1:
        raise SystemExit(f"{what}: {len(found)} candidates; set the override variable")
    return found[0]

oof_c = glob.glob(f"{TRAIN_NB}/**/oof_all.csv", recursive=True)
OOF = pick(oof_c, OOF_OVERRIDE, "oof_all.csv")
oof = pd.read_csv(OOF)
print(oof.shape); print(list(oof.columns))

ID = next((c for c in oof.columns if c.lower() in ("studyinstanceuid", "study_id", "studyuid")), None)
assert ID, "no study id column; see printed columns"
ycols = [c for c in oof.columns if c.startswith("y_")]
labels = [c[2:] for c in ycols]
assert len(labels) == 12, f"expected 12 y_ columns, got {len(labels)}: {ycols}"
if PRED_PREFIX_OVERRIDE: PP = PRED_PREFIX_OVERRIDE
else:
    PP = next((p for p in ["p_", "pred_", "prob_", "pr_", "s_", "score_"] if all(p + l in oof.columns for l in labels)), None)
    if PP is None:
        raise SystemExit("could not find prediction columns; set PRED_PREFIX_OVERRIDE. Non-y columns: "
                         + str([c for c in oof.columns if not c.startswith("y")]))
print("id:", ID, "| prediction prefix:", PP, "| fold column:", "fold" if "fold" in oof.columns else "none")
print("rows with predictions:", oof[PP + labels[0]].notna().sum(), "of", len(oof))

oof_all.csv candidates found: ['/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/oof_all.csv']
(4407, 28)
['StudyInstanceUID', 'fold', 'site_proxy', 'is_gold', 'p_ACL', 'y_ACL', 'p_MCL', 'y_MCL', 'p_Medial Meniscus', 'y_Medial Meniscus', 'p_Lateral Meniscus', 'y_Lateral Meniscus', 'p_Medial OA', 'y_Medial OA', 'p_Lateral OA', 'y_Lateral OA', 'p_PF OA', 'y_PF OA', 'p_Effusion', 'y_Effusion', 'p_Synovitis', 'y_Synovitis', "p_Baker's", "y_Baker's", 'p_Contusion', 'y_Contusion', 'p_Fracture', 'y_Fracture']
id: StudyInstanceUID | prediction prefix: p_ | fold column: fold
rows with predictions: 4407 of 4407


In [3]:
# 2. vendor / field strength / language / site columns (site_proxy is already inside oof_all.csv)
def vend(s):
    s = str(s).upper()
    if "PHILIPS" in s: return "Philips"
    if "SIEMENS" in s: return "Siemens"
    if "TOSHIBA" in s or "CANON" in s: return "Toshiba/Canon"
    if s.startswith("GE") or "GENERAL" in s: return "GE"
    return "Other"

def field(s):
    m = re.search(r"(\d+(?:\.\d+)?)\s*T\b", str(s).split("|")[-1])
    if not m: return "unknown"
    v = float(m.group(1))
    return "3T" if 2.5 <= v <= 3.5 else "1.5T" if 1.2 <= v <= 1.8 else "unknown"

meta_c = sorted(set([f"{CACHE_NB}/cache_index.csv", f"{CACHE_NB}/cache/cache_index.csv"] + glob.glob(f"{CACHE_NB}/*/cache_index.csv")))
extra = []
if META_OVERRIDE or any(os.path.exists(c) for c in meta_c):
    META = pick(meta_c, META_OVERRIDE, "cache_index.csv")
    meta = pd.read_csv(META); print(meta.shape, list(meta.columns))
    mid = next((c for c in meta.columns if c.lower() in ("studyinstanceuid", "study_id", "studyuid")), None)
    assert mid, "no study id in metadata"
    new = [c for c in meta.columns if c not in oof.columns and c != mid and re.search("vendor|manufact|site|scanner", c, re.I) and 2 <= meta[c].nunique() <= 80]
    if new:   # only columns NOT already in oof, otherwise pandas renames them _x/_y and they vanish
        oof = oof.merge(meta[[mid] + new].rename(columns={mid: ID}).drop_duplicates(ID), on=ID, how="left"); extra += new
if "site_proxy" in oof.columns:
    oof["vendor"] = oof["site_proxy"].map(lambda v: vend(str(v).split("|")[0]))
    oof["field"] = oof["site_proxy"].map(field)
    oof["vendor_field"] = oof["vendor"] + " " + oof["field"]
    print("vendor counts:", oof["vendor"].value_counts().to_dict(), "(spec expects ~ Siemens 1951, Philips 1301, GE 868, Toshiba 181)")
    print("field strength counts:", oof["field"].value_counts().to_dict(), "| site_proxy groups:", oof["site_proxy"].nunique())

# report language: detected from the text (train.csv has no language column)
_t = TRAIN_CSV_OVERRIDE or f"{DATA}/train.csv"
LANG = False
if os.path.exists(_t):
    _tr = pd.read_csv(_t)
    _tid = next((c for c in _tr.columns if c.lower() in ("studyinstanceuid", "study_id", "studyuid")), None)
    _obj = [c for c in _tr.columns if not pd.api.types.is_numeric_dtype(_tr[c]) and c != _tid]
    _rep = max(_obj, key=lambda c: _tr[c].astype(str).str.len().mean())
    try:
        from langdetect import detect, DetectorFactory
        DetectorFactory.seed = 0; HAVE_LD = True
    except Exception:
        HAVE_LD = False; print("langdetect not installed (pip install langdetect, internet ON): using script-only detection (Greek / Cyrillic / other-Latin)")
    def lang_of(t):
        t = str(t)[:1500]
        if sum("\u0370" <= c <= "\u03ff" for c in t) > 20: return "el"
        if sum("\u0400" <= c <= "\u04ff" for c in t) > 20: return "bg"
        if HAVE_LD:
            try: return detect(t)
            except Exception: return "unk"
        return "latin"
    _tr["language"] = _tr[_rep].map(lang_of)
    vc = _tr["language"].value_counts()
    _tr["language"] = _tr["language"].where(_tr["language"].isin(vc[vc >= 30].index), "other")
    oof = oof.merge(_tr[[_tid, "language"]].rename(columns={_tid: ID}).drop_duplicates(ID), on=ID, how="left")
    extra.append("language"); LANG = True
    print("report language counts:", oof["language"].value_counts().to_dict(), "(spec: tr, hr, el, bg, nl have no rules-labeller coverage)")

G = GROUP_COLS_OVERRIDE or [c for c in ["vendor", "field", "vendor_field", "language", "site_proxy"] + extra if c in oof.columns]
G = list(dict.fromkeys(G))
print("grouping columns:", G)
assert G, "no grouping columns found; set GROUP_COLS_OVERRIDE"


cache_index.csv candidates found: ['/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache/cache_index.csv']
(4407, 16) ['StudyInstanceUID', 'ok', 'site_proxy', 'err', 'present_0', 'plane_0', 'fluid_0', 'fat_0', 'present_1', 'plane_1', 'fluid_1', 'fat_1', 'present_2', 'plane_2', 'fluid_2', 'fat_2']
vendor counts: {'Siemens': 1951, 'Philips': 1301, 'GE': 905, 'Toshiba/Canon': 226, 'Other': 24} (spec expects ~ Siemens 1951, Philips 1301, GE 868, Toshiba 181)
field strength counts: {'1.5T': 2543, '3T': 1601, 'unknown': 263} | site_proxy groups: 59
langdetect not installed (pip install langdetect, internet ON): using script-only detection (Greek / Cyrillic / other-Latin)
report language counts: {'latin': 3866, 'el': 321, 'bg': 220} (spec: tr, hr, el, bg, nl have no rules-labeller coverage)
grouping columns: ['vendor', 'field', 'vendor_field', 'language', 'site_proxy']


In [4]:
# 3. per-label AUC on confident labels (same rule as the earlier yardstick) + pair loss per study
def conf(y): return (y >= 0.9) | (y <= 0.1)

def pair_loss(yb, s):
    """Per-study share of the ranking error. For a positive: fraction of negatives scored at or above it.
    For a negative: fraction of positives scored at or below it. Mean over positives = mean over negatives = 1-AUC."""
    yb = np.asarray(yb); s = np.asarray(s, float)
    pos, neg = np.sort(s[yb == 1]), np.sort(s[yb == 0])
    out = np.zeros(len(s))
    for i, (yy, ss) in enumerate(zip(yb, s)):
        if yy == 1:
            below = np.searchsorted(neg, ss, "left") + 0.5 * (np.searchsorted(neg, ss, "right") - np.searchsorted(neg, ss, "left"))
            out[i] = 1 - below / len(neg)
        else:
            below = np.searchsorted(pos, ss, "left") + 0.5 * (np.searchsorted(pos, ss, "right") - np.searchsorted(pos, ss, "left"))
            out[i] = below / len(pos)
    return out

rows, long = [], []
for l in labels:
    d = oof[[ID, "y_" + l, PP + l] + [c for c in G if c in oof.columns]].dropna(subset=["y_" + l, PP + l])
    d = d[conf(d["y_" + l])].copy()
    d["yb"] = (d["y_" + l] >= 0.9).astype(int)
    if d["yb"].nunique() < 2: continue
    d["label"] = l; d["score"] = d[PP + l]
    d["pct"] = d["score"].rank(pct=True)        # 1.0 = highest score for this label
    d["pair_loss"] = pair_loss(d["yb"], d["score"])
    a = roc_auc_score(d["yb"], d["score"])
    rows.append(dict(label=l, n=len(d), positives=int(d["yb"].sum()), AUC=round(a, 3), lost=round(1 - a, 3)))
    long.append(d[[ID, "label", "yb", "y_" + l, "score", "pct", "pair_loss"] + [c for c in G if c in d.columns]].rename(columns={"y_" + l: "y_raw"}))
tab = pd.DataFrame(rows).sort_values("lost", ascending=False)
err = pd.concat(long, ignore_index=True)
print(tab.to_string(index=False))
print("\nmacro AUC (pooled OOF over all rows with predictions):", round(tab["AUC"].mean(), 4),
      "| expect ~0.76 pooled for the full baseline; ~0.78 on fold 0 only")
if "fold" in oof.columns:
    print("per-fold macro AUC:")
    for f, dd in oof.groupby("fold"):
        aucs = []
        for l in labels:
            x = dd[["y_" + l, PP + l]].dropna(); x = x[conf(x["y_" + l])]
            yb = (x["y_" + l] >= 0.9).astype(int)
            if yb.nunique() == 2: aucs.append(roc_auc_score(yb, x[PP + l]))
        print("  fold", f, round(float(np.mean(aucs)), 4) if aucs else "n/a")
tab.to_csv(f"{OUT}/per_label_auc.csv", index=False)

           label    n  positives   AUC  lost
           PF OA 4242       1589 0.697 0.303
             MCL 4300        542 0.732 0.268
      Lateral OA 4234        714 0.734 0.266
Lateral Meniscus 4128        618 0.738 0.262
       Contusion 4101        614 0.751 0.249
         Baker's 4350       1298 0.752 0.248
 Medial Meniscus 4099       1669 0.763 0.237
        Fracture 4321        215 0.778 0.222
             ACL 4247        634 0.779 0.221
       Synovitis 4392       1011 0.780 0.220
        Effusion 3734       2049 0.822 0.178
       Medial OA 4275       1459 0.824 0.176

macro AUC (pooled OOF over all rows with predictions): 0.7625 | expect ~0.76 pooled for the full baseline; ~0.78 on fold 0 only
per-fold macro AUC:
  fold 0 0.7795
  fold 1 0.7557
  fold 2 0.7743
  fold 3 0.7663
  fold 4 0.7691


In [5]:
# 4. where do the errors cluster? AUC by vendor / site / language group (groups with >=30 studies; labels need >=5 of each class)
def macro_auc(sub):
    aucs = []
    for l, d in sub.groupby("label"):
        if d["yb"].sum() >= 5 and (1 - d["yb"]).sum() >= 5:
            aucs.append(roc_auc_score(d["yb"], d["score"]))
    return (np.mean(aucs) if aucs else np.nan), len(aucs)

overall = macro_auc(err)[0]
print("overall macro AUC on this table:", round(overall, 4))
missing = [c for c in G if c not in err.columns]
assert not missing, f"grouping columns lost on the way into the error table: {missing}"
for g in G:
    rows = []
    for v, sub in err.groupby(g):
        n_st = sub[ID].nunique()
        if n_st < 30: continue
        m, k = macro_auc(sub)
        rows.append(dict(group=v, studies=n_st, labels_scored=k, macro_AUC=round(m, 3), vs_overall=round(m - overall, 3)))
    if not rows: print(f"\n== by {g}: no group has >=30 studies ==")
    if rows:
        print(f"\n== by {g} ==")
        print(pd.DataFrame(rows).sort_values("macro_AUC").to_string(index=False))
print("\nCaveat: differences under ~0.02 are inside the seed/sampling noise of this project; a vendor effect is not a site effect (confounded).")

overall macro AUC on this table: 0.7624

== by vendor ==
        group  studies  labels_scored  macro_AUC  vs_overall
Toshiba/Canon      226             12      0.737      -0.025
      Philips     1301             12      0.752      -0.010
      Siemens     1951             12      0.756      -0.006
           GE      905             12      0.797       0.035

== by field ==
  group  studies  labels_scored  macro_AUC  vs_overall
     3T     1601             12      0.754      -0.008
   1.5T     2543             12      0.768       0.006
unknown      263             12      0.779       0.016

== by vendor_field ==
                group  studies  labels_scored  macro_AUC  vs_overall
           Philips 3T      676             12      0.741      -0.021
           Siemens 3T      792             12      0.747      -0.015
         Philips 1.5T      624             12      0.766       0.003
Toshiba/Canon unknown      129             12      0.766       0.003
         Siemens 1.5T     1159    

In [6]:
# 4c. are scores offset between sites? AUC over positive/negative pairs from the SAME group vs from DIFFERENT groups
def pair_split(col):
    rows = []
    for l, dd in err.groupby("label"):
        dd = dd.dropna(subset=[col])
        npos, nneg = int(dd["yb"].sum()), int((1 - dd["yb"]).sum())
        if npos < 20 or nneg < 20: continue
        conc_all = roc_auc_score(dd["yb"], dd["score"]) * npos * nneg
        conc_same = pairs_same = 0.0
        for _, s_ in dd.groupby(col):
            p_, n_ = s_.loc[s_["yb"] == 1, "score"].values, s_.loc[s_["yb"] == 0, "score"].values
            if len(p_) == 0 or len(n_) == 0: continue
            diff = p_[:, None] - n_[None, :]
            conc_same += (diff > 0).sum() + 0.5 * (diff == 0).sum(); pairs_same += len(p_) * len(n_)
        pairs_cross = npos * nneg - pairs_same
        if pairs_same == 0 or pairs_cross == 0: continue
        rows.append(dict(label=l, same_group_AUC=conc_same / pairs_same, cross_group_AUC=(conc_all - conc_same) / pairs_cross, share_pairs_same=pairs_same / (npos * nneg)))
    t = pd.DataFrame(rows)
    t["gap_same_minus_cross"] = t["same_group_AUC"] - t["cross_group_AUC"]
    return t

for col in [c for c in ["site_proxy", "vendor", "field", "language"] if c in err.columns]:
    t = pair_split(col)
    if t.empty: continue
    print(f"\n== pairs within the same {col} vs across different {col} ==")
    print(t.round(3).to_string(index=False))
    print("macro: same", round(t["same_group_AUC"].mean(), 4), "| cross", round(t["cross_group_AUC"].mean(), 4), "| gap", round(t["gap_same_minus_cross"].mean(), 4))
print("\nReading: a clearly positive gap means the model ranks a positive and a negative correctly more often when both come from one site/vendor/language than when they come from different ones. That can be a score offset between sites, but ALSO real prevalence or label-quality differences between them, so it is a hint, not proof. Gaps under ~0.02 are noise.")



== pairs within the same site_proxy vs across different site_proxy ==
           label  same_group_AUC  cross_group_AUC  share_pairs_same  gap_same_minus_cross
             ACL           0.798            0.778             0.045                 0.020
         Baker's           0.744            0.752             0.043                -0.008
       Contusion           0.748            0.751             0.049                -0.003
        Effusion           0.818            0.822             0.045                -0.005
        Fracture           0.791            0.778             0.038                 0.013
Lateral Meniscus           0.720            0.739             0.048                -0.019
      Lateral OA           0.751            0.733             0.040                 0.018
             MCL           0.731            0.732             0.048                -0.001
 Medial Meniscus           0.778            0.762             0.046                 0.016
       Medial OA           0.

In [7]:
# 5. how concentrated is the error for one label, and which studies are the worst?
d = err[err["label"] == LABEL].copy()
tot = d["pair_loss"].sum()
for yb, name in [(1, "positives"), (0, "negatives")]:
    x = d[d["yb"] == yb].sort_values("pair_loss", ascending=False)
    k = max(1, int(0.1 * len(x)))
    print(f"{LABEL}: worst 10% of {name} ({k} studies) carry {x['pair_loss'].head(k).sum() / x['pair_loss'].sum():.0%} of that class's ranking error")
cols = [ID, "yb", "y_raw", "score", "pct", "pair_loss"] + [c for c in G if c in d.columns]
print("\nPositives the model scored lowest (missed findings, or wrong labels):")
print(d[d.yb == 1].sort_values("score").head(15)[cols].to_string(index=False))
print("\nNegatives the model scored highest (false alarms, or wrong labels):")
print(d[d.yb == 0].sort_values("score", ascending=False).head(15)[cols].to_string(index=False))
err.sort_values(["label", "pair_loss"], ascending=[True, False]).groupby("label").head(200).to_csv(f"{OUT}/top_errors_per_label.csv", index=False)

ACL: worst 10% of positives (63 studies) carry 36% of that class's ranking error
ACL: worst 10% of negatives (361 studies) carry 30% of that class's ranking error

Positives the model scored lowest (missed findings, or wrong labels):
                                                StudyInstanceUID  yb  y_raw    score      pct  pair_loss        vendor   field          vendor_field language                        site_proxy
1.2.826.0.1.3680043.8.498.78465765647012043478539489340002371960   1   0.95 0.116625 0.001884   0.998063 Toshiba/Canon unknown Toshiba/Canon unknown    latin                TOSHIBA|Vantage|?T
1.2.826.0.1.3680043.8.498.40436886205373732563782477802719203679   1   0.95 0.127986 0.006593   0.992804 Toshiba/Canon unknown Toshiba/Canon unknown    latin                TOSHIBA|Vantage|?T
1.2.826.0.1.3680043.8.498.13269841239774089690227054060975903636   1   0.95 0.130450 0.008006   0.991420 Toshiba/Canon unknown Toshiba/Canon unknown    latin                TOSHIBA|Vantage|?

In [8]:
# 5b. does the model lean on co-occurring findings? AUC for LABEL inside subgroups defined by OTHER labels
base_auc = roc_auc_score(d["yb"], d["score"]); rows = []
for o in labels:
    if o == LABEL: continue
    x = oof[[ID, "y_" + o]].merge(d[[ID, "yb", "score"]], on=ID)
    for name, m in [("present", x["y_" + o] >= 0.9), ("absent", x["y_" + o] <= 0.1)]:
        sub_ = x[m]
        if sub_["yb"].sum() >= 20 and (1 - sub_["yb"]).sum() >= 20:
            rows.append(dict(other_label=o, status=name, n=len(sub_), positives=int(sub_["yb"].sum()),
                             AUC=round(roc_auc_score(sub_["yb"], sub_["score"]), 3),
                             mean_score_of_true_negatives=round(sub_.loc[sub_["yb"] == 0, "score"].mean(), 3)))
t = pd.DataFrame(rows)
print(f"{LABEL}: overall AUC {base_auc:.3f}. AUC inside subgroups (lowest first):")
print(t.sort_values("AUC").to_string(index=False))
print("\nMean score given to TRUE NEGATIVES by whether the other finding is present (big gap = shortcut):")
pv = t.pivot(index="other_label", columns="status", values="mean_score_of_true_negatives")
pv["gap"] = (pv["present"] - pv["absent"]).round(3)
print(pv.sort_values("gap", ascending=False).to_string())
t.to_csv(f"{OUT}/cooccurrence_{LABEL.replace(' ', '_')}.csv", index=False)

ACL: overall AUC 0.779. AUC inside subgroups (lowest first):
     other_label  status    n  positives   AUC  mean_score_of_true_negatives
       Medial OA present 1392        182 0.660                         0.311
      Lateral OA present  688        108 0.671                         0.300
       Synovitis present  939        213 0.707                         0.356
           PF OA present 1536        168 0.720                         0.286
        Effusion  absent 1649        115 0.724                         0.238
 Medial Meniscus present 1605        289 0.729                         0.319
       Contusion  absent 3384        335 0.734                         0.273
             MCL  absent 3640        407 0.747                         0.280
        Effusion present 1951        436 0.754                         0.343
         Baker's present 1251        172 0.758                         0.291
Lateral Meniscus present  596        151 0.764                         0.327
        Fractur

In [9]:
# 6. read the reports behind the worst errors: is it the label or the model?
tcs = [TRAIN_CSV_OVERRIDE] if TRAIN_CSV_OVERRIDE else [f"{DATA}/train.csv"]
if not os.path.exists(tcs[0]):
    print("train.csv not found; attach the competition data or set TRAIN_CSV_OVERRIDE"); raise SystemExit
tr = pd.read_csv(tcs[0])
print(list(tr.columns))
tid = next((c for c in tr.columns if c.lower() in ("studyinstanceuid", "study_id", "studyuid")), None)
obj = [c for c in tr.columns if not pd.api.types.is_numeric_dtype(tr[c]) and c != tid]
rep = max(obj, key=lambda c: tr[c].astype(str).str.len().mean())   # longest text column = the report
print("study id:", tid, "| report column (guess):", rep)

N = 20
worst = pd.concat([d[d.yb == 1].sort_values("score").head(N), d[d.yb == 0].sort_values("score", ascending=False).head(N)])
rev = worst[[ID, "yb", "y_raw", "score"]].merge(tr[[tid, rep]].rename(columns={tid: ID, rep: "report"}), on=ID, how="left")
KW = {"ACL": r"\bACL\b|anterior cruciate|cruciate|\bVKB\b|Kreuzb|ön çapraz|çapraz|križn|cruzado|kruisband|χιαστ|кръстн|croisé"}   # add more labels here
def snippet(txt, pat):
    txt = re.sub(r"\s+", " ", str(txt))
    mm = re.search(pat, txt, re.I) if pat else None
    if not mm: return txt[:500]
    i = max(0, mm.start() - 150); return ("..." if i else "") + txt[i:i + 600]
rev["report"] = rev["report"].map(lambda t: snippet(t, KW.get(LABEL)))
rev["my_verdict"] = ""     # fill in: label_wrong / ambiguous / real_miss
rev.insert(0, "label", LABEL)
rev.to_csv(f"{OUT}/review_{LABEL.replace(' ', '_')}.csv", index=False)
for _, r in rev.iterrows():
    print(f"[{'POS' if r.yb else 'NEG'}] y={r.y_raw:.2f} score={r.score:.3f}\n   {r.report}\n")
print("Saved review sheet. Mark each case label_wrong / ambiguous / real_miss, then count them.")

['StudyInstanceUID', 'Report', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
study id: StudyInstanceUID | report column (guess): Report
[POS] y=0.95 score=0.117
   ...ilage of patella, major at medial facet and patellar ridge, reflecting grade 3 chondromalacia. Laxity of medial patellar retinaculum > The contour of ACL and MCL in a straight line and increase signal intensity inside on F/S T2WI, in favor of gade 1 injury > The contour of PCL and LCL within normal > No evidence of tear is noted in either medial or lateral meniscus. > Both medial & lateral collateral ligaments appear to be within normal. > Increased fat stranding and minimal fluid accumulation located along the pes anserinus, R/O paratendinitis. > Tendinosis of quadriceps tendon. Impression: 1

[POS] y=0.95 score=0.128
   ...cal root, in favor of tear. > The contour of PCL is not in a straight line and increase signal i

## How to read the result
- **label_wrong** (report says the opposite of the label): the loss is the labeller's. A better model won't help; better labels or label-aware weighting will.
- **ambiguous** (report vague, finding probably borderline): little to gain.
- **real_miss** (report clearly states the finding, model disagrees): model-limited. Look at series chosen, slice coverage, vendor/site group, image quality for these.
- Change `LABEL` in cell 0 and re-run cells 5-6 for each label. Start with ACL, MCL, Medial Meniscus, Lateral Meniscus (model-limited per the spec) and Effusion (the known 15-case label mismatch).
Output files are in `/kaggle/working/error_analysis/` (commit with Save Version to keep them).